# NYC Taxi Trip Duration EDA

Orginal data set:
https://www.kaggle.com/datasets/yasserh/nyc-taxi-trip-duration?utm_source=chatgpt.com

Author of orginal dataset:
https://www.kaggle.com/yasserh

#### About dataset:
The competition dataset is based on the 2016 NYC Yellow Cab trip record data made available in Big Query on Google Cloud Platform. The data was originally published by the NYC Taxi and Limousine Commission (TLC). The data was sampled and cleaned for the purposes of this playground competition. Based on individual trip attributes, participants should predict the duration of each trip in the test set.

#### Columns
- id - a unique identifier for each trip

- vendor_id - a code indicating the provider associated with the trip record

- pickup_datetime - date and time when the meter was engaged

- dropoff_datetime - date and time when the meter was disengaged

- passenger_count - the number of passengers in the vehicle (driver entered value)

- pickup_longitude - the longitude where the meter was engaged

- pickup_latitude - the latitude where the meter was engaged

- dropoff_longitude - the longitude where the meter was disengaged

- dropoff_latitude - the latitude where the meter was disengaged

- store_and_fwd_flag - This flag indicates whether the trip record was held in vehicle memory before sending to the vendor because the vehicle did not have a connection to the server - Y=store and forward; N=not a store and forward trip

- trip_duration - duration of the trip in seconds

# Goals of analis:

- Understanding the relationships between columns

- Preparing data for ML regression

In [ ]:
# Imports

import pandas as pd
import numpy as np
import re
from IPython.display import Markdown
from IPython.display import Markdown, display
# from itables import show
import plotly.express as px
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from collections import Counter

## 1. Technial Overview

Set the number to a readable format (not Scientific Notation (-6.133553e+01))

In [ ]:
pd.set_option("display.float_format", "{:.6f}".format)

### 1.1 Sample

In [ ]:
raw_df = pd.read_csv('data/NYC.csv')
raw_df.sample(5)

### 1.2 Basic data info

In [ ]:
info_df = pd.DataFrame({
    'Column': raw_df.columns,
    "Duplicates": raw_df.duplicated().sum(),
    'Non-null': raw_df.notnull().sum().values,
    'Missing Values': raw_df.isna().sum().values,
    'Percentage of Missing Values': (raw_df.isna().sum()*100/raw_df.notnull().sum()).values,
    'Unique Values': raw_df.nunique().values,
    'Data Type': raw_df.dtypes.values,
})

info_df

### 1.3 Describe of numeric columns

In [ ]:
raw_df.describe().T

#### Sumarry of Technial Overview

- Kolumna id nie wnosi zadnej wartośc ani do EDA, ani do modelu. posiada 1458644 wartości unikatowe tyle co rekordów.

- Istnieją 2 kolumny binarne 'vendor id' oraz 'store_and_fwd_flag', przyli kandydaci do hot one encoding

- Passenger_count posiadają 10 wartości unikatowych od 0 do 9, wartość zero może być myląca ale wcale nie niemmozliwa (delivery, cargo logistic etc)

- Kolumny takie jak:
    - pickup_longitude	(min: -121.933342, max: -61.335529)
    - pickup_latitude	(min: 34.359695, max: 51.881084 )
    - dropoff_longitude (min -121.933304, max: -61.335529)
    - dropoff_latitude  (min: 32.181141, max: 43.921028)

    wymagają naprawy w daleszej częsci ponieważ, granice nowego jorku wedłóg NYC Borough Boundary Metadata (official NYC data) wynosza kolejno:

    - West longitude: -74.257159
    - East longitude: -73.699215
    - North latitude: 40.915568
    - South latitude: 40.496010

    Biorąc pod uwagę że kursy mogly być robione nawet w okolicach przedmieśc gołym okiem widać że maxymalne i minimalne wartości sa przesadzone.

- Koluumna 'trip_duration' wymaga ewidentnej naprawy i ewntualnego ujednolicenia jednostek. Teoretycznie wygodnie było by aby kolumna ta była wyrazona w sekundach. Wartość minimalna równa 1 wskazuje albo błąd, albo podawane w innych jednostkach czasu np: godzinach. Skolei warość maksymalna wartość 3526282 czas zbliżony do 58 minut wyrażony w milisekundach (co jest mozliwe)

#### Summary of Technical Overview
- The 'id' column provides no meaningful value for either EDA or modeling purposes. It contains 1,458,644 unique values — exactly the same number as the dataset records — making it purely an identifier feature.

- There are two binary/categorical columns: vendor_id and store_and_fwd_flag, making them potential candidates for categorical encoding (e.g. one-hot encoding).

- passenger_count contains 10 unique values ranging from 0 to 9. While a value of 0 may initially appear suspicious, it is not necessarily impossible in real-world scenarios (e.g. delivery rides, cargo logistics, repositioning trips, etc.).

- The following geolocation columns:

    - pickup_longitude (min: -121.933342, max: -61.335529)
    - pickup_latitude (min: 34.359695, max: 51.881084)
    - dropoff_longitude (min: -121.933304, max: -61.335529)
    - dropoff_latitude (min: 32.181141, max: 43.921028)

    clearly require further validation and cleaning. According to the official NYC Borough Boundary Metadata, the approximate geographic boundaries of New York City are:

    - West longitude: -74.257159
    - East longitude: -73.699215
    - North latitude: 40.915568
    - South latitude: 40.496010

    Although some taxi rides may legitimately extend beyond NYC city limits into suburban areas, the observed minimum and maximum coordinate values appear heavily exaggerated and likely contain corrupted or invalid records.

- The 'trip_duration' column requires further investigation and possible unit standardization. Ideally, trip duration should be represented consistently in seconds. However, the absence of explicit unit information introduces uncertainty regarding how the values were originally recorded.

    Several real-world scenarios could potentially explain the observed inconsistencies:

    - different dispatch or metering systems may have stored duration values using different units (seconds, minutes, milliseconds, etc.)
    - some systems may have displayed human-readable time formats while internally storing values differently
    - manual adjustments, rounding, export issues, or legacy system integrations could also contribute to inconsistent representations

    The minimum value of 1 and the maximum value of 3526282 appear suspicious and require additional validation. At this stage, it is not yet possible to determine whether these values represent:
    
    - corrupted or broken trip records
    - extreme outliers
    - or inconsistently recorded time units

    Further investigation through distribution analysis, quantile inspection, and outlier visualization will be necessary before applying any cleaning strategy.